In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not on colab')

# Movie Recommender System using MovieLens data (train)
Author: Michael Fitiavana Randriamifidy

Created: 26 December 2025

Last Updated: 10 January 2025

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

In [ ]:
DATA_SIZE_DICT= {
    0: "ml-latest-small",
    1: "ml-25m ",
    2: "ml-32m",
}
DATA_SIZE_STR = DATA_SIZE_DICT[2]

# Setting up

In [ ]:
import importlib
sys.modules["imp"] = importlib
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

## Setting up Notebooks

In [ ]:
from typing import Optional, Callable, Dict, Any, List

def download_and_extract_movielens(
                            data_folders:List[str],
                            extracted_data_folder:str='./data',
                            data_size_str:str='ml-latest-small',
                            only_extract:bool=True
                            ):
    pattern = re.compile(rf"{data_size_str}\.zip$")
    for data_folder in data_folders:
        matches = [p for p in Path(data_folder).iterdir() if pattern.match(p.name)]
        if matches:
            print("Found zip file(s):", str(matches[-1]))
            break
    if not  matches:
        print(f"Downloading:{data_size_str}.zip from https://files.grouplens.org/datasets/movielens/{data_size_str}.zip")
        !curl -C - -O --output-dir "{data_folder}" "https://files.grouplens.org/datasets/movielens/{data_size_str}.zip"
    print(f"Extract to {extracted_data_folder}")
    !unzip -n "{data_folder}/{data_size_str}.zip" -d "{extracted_data_folder}"


    pattern = re.compile(rf"{data_size_str}\.zip.md5$")
    for data_folder in data_folders:
        matches = [p for p in Path(data_folder).iterdir() if pattern.match(str(p))]
        if matches:
            print("Found zip.md5 file(s):", str(matches[-1]))
            break
    if not  matches:
        print(f"Downloading:{data_size_str}.zip.md5 from https://files.grouplens.org/datasets/movielens/{data_size_str}.zip.md5")
        !curl -C - -O --output-dir "{data_folder}"  "https://files.grouplens.org/datasets/movielens/{data_size_str}.zip.md5"

def is_kaggle() -> bool:
    """
    Detect if the code is running on Kaggle.

    Returns:
        bool: True if running on Kaggle, False otherwise.
    """
    return "KAGGLE_URL_BASE" in os.environ


def is_colab() -> bool:
    """
    Detect if the code is running on Google Colab.

    Returns:
        bool: True if running on Google Colab, False otherwise.
    """
    try:
        import google.colab
        return True
    except ImportError:
        return False


def set_project(
    fun_todo_kaggle: Optional[Callable[..., Any]] = None,
    fun_todo_colab: Optional[Callable[..., Any]] = None,
    fun_todo_local: Optional[Callable[..., Any]] = None,
    kwargs_kaggle: Optional[Dict[str, Any]] = None,
    kwargs_colab: Optional[Dict[str, Any]] = None,
    kwargs_local: Optional[Dict[str, Any]] = None
) -> Optional[Any]:
    """
    Execute platform-specific setup functions based on the current environment.

    This function detects whether code is running on Kaggle, Google Colab, or locally,
    and executes the corresponding setup function with the provided keyword arguments.

    Args:
        fun_todo_kaggle: Optional function to execute on Kaggle.
        fun_todo_colab: Optional function to execute on Google Colab.
        fun_todo_local: Optional function to execute locally.
        kwargs_kaggle: Keyword arguments to pass to the Kaggle function.
        kwargs_colab: Keyword arguments to pass to the Colab function.
        kwargs_local: Keyword arguments to pass to the local function.

    Returns:
        The return value of the executed function, or None if no function is provided
        for the detected environment.
    """
    # Initialize kwargs dictionaries if None
    kwargs_kaggle = kwargs_kaggle or {}
    kwargs_colab = kwargs_colab or {}
    kwargs_local = kwargs_local or {}

    # Conditional commands
    if is_kaggle():
        print("Running on Kaggle")
        if fun_todo_kaggle is not None:
            return fun_todo_kaggle(**kwargs_kaggle)
    elif is_colab():
        print("Running on Google Colab")
        if fun_todo_colab is not None:
            return fun_todo_colab(**kwargs_colab)
    else:
        print("Running locally")
        if fun_todo_local is not None:
            return fun_todo_local(**kwargs_local)

    return None

def is_drive_mounted():
    return os.path.exists("/content/drive/MyDrive")


def fun_todo_kaggle(**kwargs):
    data_size_str = kwargs['data_size_str'] if "data_size_str" in kwargs else "latest-small"
    project_dir_abs = kwargs.get("project_dir_abs", Path.cwd())
    # os.chdir(project_dir_abs)
    REMOTE_PROJECT_PATH = Path(f"/kaggle/working/movie-recommender-system-{data_size_str}")
    PROJECT_PATH = Path("/kaggle/temp/")
    DATA_FOLDER = PROJECT_PATH / "download/data"
    EXTRACTED_DATA_FOLDER = Path("/kaggle/input/movielens-dataset/")
    PROJECT_PATH.mkdir(exist_ok=True, parents=True)
    EXTRACTED_DATA_FOLDER.mkdir(exist_ok=True, parents=True)
    REMOTE_PROJECT_PATH.mkdir(exist_ok=True, parents=True)
    DATA_FOLDER.mkdir(exist_ok=True, parents=True)
    !pwd
    !ls $REMOTE_PROJECT_PATH -R | head -20
    if project_dir_abs.as_posix() not in sys.path:
        sys.path.append(project_dir_abs.as_posix())
    print(sys.path)
    return REMOTE_PROJECT_PATH, PROJECT_PATH, DATA_FOLDER, EXTRACTED_DATA_FOLDER

def fun_todo_colab(**kwargs):
    if is_drive_mounted():
        print("Google Drive is already mounted")
    else:
        raise "Mount Drive"

    data_size_str = kwargs['data_size_str'] if "data_size_str" in kwargs else "latest-small"
    project_dir_abs = kwargs.get("project_dir_abs", Path.cwd())
    # os.chdir(project_dir_abs)
    REMOTE_PROJECT_PATH = Path(project_dir_abs) / f"working/movie-recommender-system-{data_size_str}"
    PROJECT_PATH =  Path(project_dir_abs) / f"temp/movie-recommender-system-{data_size_str}"
    DATA_FOLDER = Path("./downloads/data")
    EXTRACTED_DATA_FOLDER = Path("./data")
    PROJECT_PATH.mkdir(exist_ok=True, parents=True)
    EXTRACTED_DATA_FOLDER.mkdir(exist_ok=True, parents=True)
    REMOTE_PROJECT_PATH.mkdir(exist_ok=True, parents=True)
    DATA_FOLDER.mkdir(exist_ok=True, parents=True)
    !pwd
    !ls $REMOTE_PROJECT_PATH -R | head -20

    if project_dir_abs.as_posix() not in sys.path:
        sys.path.append(project_dir_abs.as_posix())
    print(sys.path)
    download_and_extract_movielens(
        [DATA_FOLDER, project_dir_abs / DATA_FOLDER],
        EXTRACTED_DATA_FOLDER,
        data_size_str = DATA_SIZE_STR)
    return REMOTE_PROJECT_PATH, PROJECT_PATH, DATA_FOLDER, EXTRACTED_DATA_FOLDER

def fun_todo_local(**kwargs):
    data_size_str = kwargs.get('data_size_str', 'latest-small')
    project_dir_abs = kwargs.get("project_dir_abs", Path.cwd())
    os.chdir(project_dir_abs)
    REMOTE_PROJECT_PATH = Path(project_dir_abs) / f"working/movie-recommender-system-{data_size_str}"
    PROJECT_PATH =  Path(project_dir_abs) / f"temp/movie-recommender-system-{data_size_str}"
    DATA_FOLDER = Path("./downloads/data")
    EXTRACTED_DATA_FOLDER = Path("./data")
    PROJECT_PATH.mkdir(exist_ok=True, parents=True)
    EXTRACTED_DATA_FOLDER.mkdir(exist_ok=True, parents=True)
    REMOTE_PROJECT_PATH.mkdir(exist_ok=True, parents=True)
    DATA_FOLDER.mkdir(exist_ok=True, parents=True)
    !pwd
    !ls $REMOTE_PROJECT_PATH -R | head -20
    if project_dir_abs.as_posix() not in sys.path:
        sys.path.append(project_dir_abs.as_posix())
    print(sys.path)

    download_and_extract_movielens(
        [DATA_FOLDER, project_dir_abs / DATA_FOLDER],
        EXTRACTED_DATA_FOLDER,
        data_size_str = DATA_SIZE_STR)
    return REMOTE_PROJECT_PATH, PROJECT_PATH, DATA_FOLDER, EXTRACTED_DATA_FOLDER


In [ ]:
if Path.cwd().name == "notebooks":
    PROJECT_DIR_ABS = Path.cwd().parent
else:
    PROJECT_DIR_ABS = Path.cwd()
print(f"MovieLens data:{DATA_SIZE_STR}")
kwargs = {"data_size_str":DATA_SIZE_STR}
kwargs_kaggle = {**kwargs, **{'project_dir_abs':Path(f"/kaggle/input/movie-recommender-system-{DATA_SIZE_STR}")}}
kwargs_colab = {**kwargs, **{'project_dir_abs':Path(f"/content/drive/MyDrive/Labs/ml_scale/movie-recommender-system")}}
kwargs_local = {**kwargs, **{'project_dir_abs':PROJECT_DIR_ABS}}
(
    REMOTE_PROJECT_PATH,
    PROJECT_PATH,
    DATA_FOLDER,
    EXTRACTED_DATA_FOLDER
    ) = set_project(fun_todo_kaggle,
                    fun_todo_colab,
                    fun_todo_local,
                    kwargs_kaggle=kwargs_kaggle,
                    kwargs_colab=kwargs_colab,
                    kwargs_local=kwargs_local)

ratings_path = EXTRACTED_DATA_FOLDER / DATA_SIZE_STR / "ratings.csv"
movies_path = EXTRACTED_DATA_FOLDER / DATA_SIZE_STR / "movies.csv"
PLOT_FOLDER = PROJECT_PATH / "plots"
MODEL_FOLDER = PROJECT_PATH / "models"
REMOTE_MODEL_FOLDER = REMOTE_PROJECT_PATH / "models"

## Import

In [ ]:
from recommender.data.csr import (
    get_nrows_csr
)

from recommender.data.ratings import (
    parse_with_feature_randomly_train_test_split_data_csr
)

from recommender.plots.plotting import (
    plot_ratings_csr_publication,
    save_plot,
)

from recommender.models.train import (
    train_recommender_als,
)

# Load-Split Datasets

In [ ]:
(
    user_id_to_idx, user_idx_to_id, movie_id_to_idx, movie_idx_to_id,
    feature_idx_to_id, feature_id_to_idx,
    ratings_by_user_train_csr, ratings_by_movie_train_csr,
    ratings_by_user_test_csr, ratings_by_movie_test_csr,
    features_by_movie_csr, movies_by_feature_csr, movies_title
)= parse_with_feature_randomly_train_test_split_data_csr(ratings_path, movies_path, split=0.9, seed=42)
assert get_nrows_csr(ratings_by_user_train_csr) == get_nrows_csr(ratings_by_user_test_csr)
assert get_nrows_csr(ratings_by_movie_train_csr) == get_nrows_csr(ratings_by_movie_test_csr) == get_nrows_csr(features_by_movie_csr)

## Power Law: Train

### Users

In [ ]:
# power law Users
type_vector = "Users"
datasplit_to = "train"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "

plot_folder = PLOT_FOLDER
filename = f"power-law-{datasplit_to}-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)
fig, ax = plt.subplots(1, figsize=(6,6))

plot_ratings_csr_publication(ratings_by_user_train_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=False,
                            color='C0',
                            fig_ax=(fig, ax))
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

### Movies

In [ ]:
# power law Movies
type_vector = "Movies"
datasplit_to = "train"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "

plot_folder = PLOT_FOLDER
filename = f"power-law-{datasplit_to}-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)
fig, ax = plt.subplots(1, figsize=(6,6))

plot_ratings_csr_publication(ratings_by_movie_train_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=False,
                            color='C1',
                            fig_ax=(fig, ax))
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

## Power Law: Test

### Users

In [ ]:
# power law Users
type_vector = "Users"
datasplit_to = "test"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "
plot_folder = PLOT_FOLDER
filename = f"power-law-{datasplit_to}-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)
fig, ax = plt.subplots(1, figsize=(6,6))

plot_ratings_csr_publication(ratings_by_user_test_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=False,
                            color='C0',
                            fig_ax=(fig, ax))
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

### Movies

In [ ]:
# power law Movies
type_vector = "Movies"
datasplit_to = "test"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "

plot_folder = PLOT_FOLDER
filename = f"power-law-{datasplit_to}-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)
fig, ax = plt.subplots(1, figsize=(6,6))

plot_ratings_csr_publication(ratings_by_movie_test_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=False,
                            color='C1',
                            fig_ax=(fig, ax))
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

# Training

## Training Bias Only

In [ ]:
bias_only = True
save_models = True

train_different_models_kwargs = {
    'precision': 1,
    'bias_penality': 1e-3,
    'vector_penality_range': [0.1, 1, 10],
    'K_range': [2, 4, 8, 16],
    'max_iter': 50
}

for K in train_different_models_kwargs['K_range']:
    for vector_penality in train_different_models_kwargs['vector_penality_range']:
        train_kwargs = {
        'precision': train_different_models_kwargs['precision'],
        'bias_penality': train_different_models_kwargs['bias_penality'],
        'vector_penality': vector_penality,
        'K': K,
        'max_iter': train_different_models_kwargs['max_iter']
    }
        _ = train_recommender_als(
            ratings_by_user_train_csr,
            ratings_by_movie_train_csr,
            ratings_by_user_test_csr,
            ratings_by_movie_test_csr,
            items_ids = movie_idx_to_id,
            feature_ids = feature_idx_to_id,
            model_folder=MODEL_FOLDER,
            data_size_str=DATA_SIZE_STR,
            bias_only=bias_only,
            save=save_models,
            **train_kwargs
            )
    print("-"*50)

## Train with Embeddings

In [ ]:
bias_only = False
with_features = False
save_models = True

train_different_models_kwargs = {
    'precision': 1,
    'bias_penality': 1e-3,
    'vector_penality_range': [0.1, 1, 10],
    'K_range': [32, 16, 8, 4, 2],
    'max_iter': 50
}

training_function_kwargs = {
        'patience': 10,
        'min_delta': 1e-3,
        'is_early_stopping':True,
    }
for K in train_different_models_kwargs['K_range']:
    for vector_penality in train_different_models_kwargs['vector_penality_range']:
        train_kwargs = {
        'precision': train_different_models_kwargs['precision'],
        'bias_penality': train_different_models_kwargs['bias_penality'],
        'vector_penality': vector_penality,
        'K': K,
        'max_iter': train_different_models_kwargs['max_iter']
    }

        _ = train_recommender_als(
            ratings_by_user_train_csr,
            ratings_by_movie_train_csr,
            ratings_by_user_test_csr,
            ratings_by_movie_test_csr,
            features_by_movie_csr,
            movies_by_feature_csr,
            items_ids = movie_idx_to_id,
            feature_ids = feature_idx_to_id,
            model_folder=MODEL_FOLDER,
            data_size_str=DATA_SIZE_STR,
            with_features=with_features,
            bias_only=bias_only,
            save=save_models,
            seed=42,
            training_function_kwargs=training_function_kwargs,
            **train_kwargs
        )
        print("="*50)
    print("="*50)


## Train with Features

In [ ]:
with_features = True
save_models = True

train_different_models_kwargs = {
    'precision': 1,
    'bias_penality': 1e-3,
    'vector_penality_range': [0.1, 1, 10],
    'K_range': [32, 16, 8, 4, 2],
    'max_iter': 50,
}
training_function_kwargs = {
        'patience': 10,
        'min_delta': 1e-3,
        'is_early_stopping':True,
    }
for K in train_different_models_kwargs['K_range']:
    for vector_penality in train_different_models_kwargs['vector_penality_range']:
        train_kwargs = {
        'precision': train_different_models_kwargs['precision'],
        'bias_penality': train_different_models_kwargs['bias_penality'],
        'vector_penality': vector_penality,
        'K': K,
        'max_iter': train_different_models_kwargs['max_iter']
    }

        _ = train_recommender_als(
            ratings_by_user_train_csr,
            ratings_by_movie_train_csr,
            ratings_by_user_test_csr,
            ratings_by_movie_test_csr,
            features_by_movie_csr,
            movies_by_feature_csr,
            items_ids = movie_idx_to_id,
            feature_ids = feature_idx_to_id,
            model_folder=MODEL_FOLDER,
            data_size_str=DATA_SIZE_STR,
            bias_only=False,
            with_features=with_features,
            save=save_models,
            seed=42,
            training_function_kwargs=training_function_kwargs,
            **train_kwargs
        )
        print("="*50)
    print("="*50)


# Copy latest to REMOTE_PROJECT_PATH

In [ ]:
from recommender.utils.helpers import copy_folder

In [ ]:
copy_folder(
    MODEL_FOLDER,
    REMOTE_MODEL_FOLDER,
    include_patterns=[r"(?=.*best_nepochs).*-latest.joblib$"],
)